# CARMA — Grip Intensity Deformer
MediaPipe + Deformer lifting pipeline for continuous hand grasp intensity estimation (0 = open, 1 = closed fist).

## Setup
Mount Drive and install dependencies. We run this every session before anything else.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'mediapipe', 'torch', 'torchvision',
                       'matplotlib', 'Pillow', 'tqdm'])

import mediapipe as mp
import torch
print(f'MediaPipe : {mp.__version__}')
print(f'PyTorch   : {torch.__version__}')


## Paths
Define where the dataset, checkpoints, and MediaPipe cache live on Drive. Update these as your folder structure is different.

In [ ]:
from pathlib import Path

DATA_ROOT = Path('/content')
CKPT_DIR  = Path('/content/drive/MyDrive/Agam_Deformer/carma_mp_ckpts')
CACHE_DIR = Path('/content/drive/MyDrive/Agam_Deformer/mp_cache')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

XYZ_PATH = DATA_ROOT / 'training_xyz.json'
RGB_DIR  = DATA_ROOT / 'training' / 'rgb'
MP_CACHE = CACHE_DIR / 'mp_landmarks.npy'
MP_MASK  = CACHE_DIR / 'mp_valid_mask.npy'

for p in [XYZ_PATH, RGB_DIR]:
    print('OK  ' if p.exists() else 'MISSING  ', p)


## Imports
Load all libraries and download the MediaPipe hand landmarker model if it's not already in `/content/`. Also sets the random seeds and confirms the GPU is available.

In [ ]:
import json, math, random, glob, time, collections
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import urllib.request, os

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

MODEL_PATH = '/content/hand_landmarker.task'
if not os.path.exists(MODEL_PATH):
    print('Downloading hand landmarker model...')
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task',
        MODEL_PATH
    )

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', 'Go to Runtime -> Change runtime type -> T4 GPU'
print(f'Device: {device} ({torch.cuda.get_device_name(0)})')


## Openness metric
Defines how we measure hand openness from 3D keypoints. We compute the flexion angle at each finger joint — straight fingers give low flexion, curled fingers give high flexion. The average across all 10 joint triples becomes the raw openness score.

In [ ]:
BEND_TRIPLES = [
    (1, 2, 3), (2, 3, 4),
    (5, 6, 7), (6, 7, 8),
    (9, 10, 11), (10, 11, 12),
    (13, 14, 15), (14, 15, 16),
    (17, 18, 19), (18, 19, 20),
]

OPEN_FLEX_RAD   = math.radians(10.0)
CLOSED_FLEX_RAD = math.radians(90.0)
FLEX_RANGE      = CLOSED_FLEX_RAD - OPEN_FLEX_RAD


def _angle_np(a, b, c):
    u, v = a - b, c - b
    nu, nv = np.linalg.norm(u), np.linalg.norm(v)
    if nu < 1e-7 or nv < 1e-7:
        return math.pi
    return float(np.arccos(np.clip(np.dot(u, v) / (nu * nv), -1.0, 1.0)))


def openness_np(kp):
    scores = []
    for a, b, c in BEND_TRIPLES:
        flex = math.pi - _angle_np(kp[a], kp[b], kp[c])
        scores.append((flex - OPEN_FLEX_RAD) / FLEX_RANGE)
    return float(np.clip(np.mean(scores), 0.0, 1.0))


def openness_torch(kp):
    vals = []
    for a, b, c in BEND_TRIPLES:
        u = kp[:, a] - kp[:, b]
        v = kp[:, c] - kp[:, b]
        cos = F.cosine_similarity(u, v, dim=-1, eps=1e-7).clamp(-1+1e-6, 1-1e-6)
        flex = math.pi - torch.acos(cos)
        vals.append((flex - OPEN_FLEX_RAD) / FLEX_RANGE)
    return torch.stack(vals, dim=1).mean(dim=1).clamp(0.0, 1.0)


## Load FreiHAND GT and compute labels
Loads the ground truth 3D keypoints from FreiHAND and computes a grasp intensity label for each sample. Raw scores are percentile-stretched so the labels span the full 0–1 range across the dataset.

In [ ]:
print('Loading training_xyz.json...')
with open(XYZ_PATH) as f:
    xyz_all = json.load(f)
xyz_np = np.array(xyz_all, dtype=np.float32)
N = len(xyz_np)
print(f'{N:,} samples, shape {xyz_np.shape}')

raw_scores = np.array(
    [openness_np(kp - kp[0:1]) for kp in xyz_np], dtype=np.float32
)

STRETCH_LO  = float(np.percentile(raw_scores, 2))
STRETCH_HI  = float(np.percentile(raw_scores, 98))
STRETCH_RNG = max(STRETCH_HI - STRETCH_LO, 1e-6)

def stretch_np(r):
    return float(np.clip((r - STRETCH_LO) / STRETCH_RNG, 0.0, 1.0))

def stretch_torch(r):
    return ((r - STRETCH_LO) / STRETCH_RNG).clamp(0.0, 1.0)

intensity_gt = np.array([stretch_np(s) for s in raw_scores], dtype=np.float32)

print(f'Raw scores      : min={raw_scores.min():.3f}  max={raw_scores.max():.3f}')
print(f'Stretched labels: min={intensity_gt.min():.3f}  max={intensity_gt.max():.3f}')
print(f'Stretch params  : lo={STRETCH_LO:.4f}  hi={STRETCH_HI:.4f}')


## Unzip dataset
Only needed the first time in a new session. Unzips the FreiHAND dataset from Drive to local `/content/` storage so MediaPipe can read the images quickly.

In [ ]:
import os

ZIP_PATH = '/content/drive/MyDrive/Agam_Deformer/freihand/FreiHAND_pub_v2.zip'

print('Unzipping FreiHAND...')
os.system(f'unzip -q "{ZIP_PATH}" -d /content/')
os.system('ls /content/')


## MediaPipe preprocessing
Runs MediaPipe Hands on every image in the dataset and saves the 21×3 landmarks to Drive as a cache. This only runs once — every future session loads the cached `.npy` files instead.

In [ ]:
def run_mediapipe_preprocessing(rgb_dir, n_samples, cache_path, mask_path):
    if cache_path.exists() and mask_path.exists():
        print(f'Loading cached landmarks from {cache_path}...')
        landmarks  = np.load(str(cache_path))
        valid_mask = np.load(str(mask_path))
        print(f'{valid_mask.sum():,} / {n_samples:,} valid detections')
        return landmarks, valid_mask

    print(f'Running MediaPipe on {n_samples:,} images...')
    landmarks  = np.full((n_samples, 21, 3), np.nan, dtype=np.float32)
    valid_mask = np.zeros(n_samples, dtype=bool)

    base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
    hand_options = mp_vision.HandLandmarkerOptions(
        base_options=base_options,
        num_hands=1,
        min_hand_detection_confidence=0.3,
    )
    detector = mp_vision.HandLandmarker.create_from_options(hand_options)

    for i in tqdm(range(n_samples), desc='MediaPipe'):
        img_path = rgb_dir / f'{i:08d}.jpg'
        if not img_path.exists():
            continue
        try:
            mp_img = mp.Image.create_from_file(str(img_path))
            result = detector.detect(mp_img)
            if result.hand_landmarks:
                lm = result.hand_landmarks[0]
                for j, l in enumerate(lm):
                    landmarks[i, j] = [l.x, l.y, l.z]
                valid_mask[i] = True
        except Exception:
            continue

    detector.close()
    n_valid = valid_mask.sum()
    print(f'Detection rate: {n_valid:,} / {n_samples:,} ({100*n_valid/n_samples:.1f}%)')
    np.save(str(cache_path), landmarks)
    np.save(str(mask_path), valid_mask)
    return landmarks, valid_mask


mp_landmarks, mp_valid = run_mediapipe_preprocessing(RGB_DIR, N, MP_CACHE, MP_MASK)
n_valid = int(mp_valid.sum())
print(f'Valid samples: {n_valid:,}')

sample_valid_idx = np.where(mp_valid)[0][42]
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
lm = mp_landmarks[sample_valid_idx]
ax.scatter(lm[:, 0], 1 - lm[:, 1], s=40, c='dodgerblue', zorder=3)
skeleton = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),(0,9),(9,10),(10,11),(11,12),
            (0,13),(13,14),(14,15),(15,16),(0,17),(17,18),(18,19),(19,20),(5,9),(9,13),(13,17)]
for a, b in skeleton:
    ax.plot([lm[a,0], lm[b,0]], [1-lm[a,1], 1-lm[b,1]], 'gray', lw=1)
ax.set(title=f'MediaPipe detection (sample {sample_valid_idx})', aspect='equal')
ax.invert_xaxis()
plt.tight_layout()
plt.show()


## Normalize landmarks
Converts MediaPipe's raw image-space coordinates to root-relative, scale-normalized form. The same normalization is applied to the GT keypoints so both inputs are in the same coordinate space when training.

In [ ]:
def normalize_mp_landmarks(lm):
    lm = lm.copy()
    lm = lm - lm[0:1]
    scale = np.linalg.norm(lm[9])
    if scale > 1e-7:
        lm = lm / scale
    return lm


mp_normed = np.full_like(mp_landmarks, np.nan)
for i in np.where(mp_valid)[0]:
    mp_normed[i] = normalize_mp_landmarks(mp_landmarks[i])

xyz_normed = np.zeros_like(xyz_np)
for i in range(N):
    kp = xyz_np[i] - xyz_np[i, 0:1]
    scale = np.linalg.norm(kp[9])
    xyz_normed[i] = kp / scale if scale > 1e-7 else kp

print(f'MP normed x range : [{np.nanmin(mp_normed[:,:,0]):.2f}, {np.nanmax(mp_normed[:,:,0]):.2f}]')
print(f'GT normed x range : [{xyz_normed[:,:,0].min():.2f}, {xyz_normed[:,:,0].max():.2f}]')


## Dataset
Groups the 4 augmented background versions of each FreiHAND pose into a clip of T=4 frames. These pseudo-sequences give the Temporal Transformer something to attend across. The dataset class also handles random 3D rotation and jitter augmentation during training.

In [ ]:
T_FRAMES = 4

class MPLiftDataset(Dataset):
    def __init__(self, base_indices, mp_normed, xyz_normed,
                 intensity_gt, n_total, augment=False):
        self.n_total      = n_total
        self.n_base       = n_total // T_FRAMES if n_total > 32560 else n_total
        self.mp_normed    = mp_normed
        self.xyz_normed   = xyz_normed
        self.intensity_gt = intensity_gt
        self.augment      = augment
        self.items        = []

        for i in base_indices:
            aug_idxs = [i + k * self.n_base for k in range(T_FRAMES)] if n_total > 32560 else [i] * T_FRAMES
            if all(not np.any(np.isnan(mp_normed[j])) for j in aug_idxs):
                self.items.append((i, aug_idxs))

        print(f'  {len(self.items):,} valid clips from {len(base_indices):,} base poses')

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        base_i, aug_idxs = self.items[idx]
        mp_clip = np.stack([self.mp_normed[j] for j in aug_idxs], axis=0).astype(np.float32)
        kp_gt   = self.xyz_normed[base_i].astype(np.float32)
        label   = float(self.intensity_gt[base_i])

        if self.augment:
            angle = random.uniform(-math.pi / 6, math.pi / 6)
            c, s  = math.cos(angle), math.sin(angle)
            Ry    = np.array([[c,0,s],[0,1,0],[-s,0,c]], dtype=np.float32)
            mp_clip = np.einsum('tji,ki->tjk', mp_clip.reshape(T_FRAMES,21,3), Ry).reshape(T_FRAMES, 21, 3)
            kp_gt   = kp_gt @ Ry.T
            mp_clip += np.random.randn(*mp_clip.shape).astype(np.float32) * 0.015

        return {
            'mp_clip': torch.tensor(mp_clip, dtype=torch.float32),
            'kp_gt'  : torch.tensor(kp_gt,   dtype=torch.float32),
            'label'  : torch.tensor(label,    dtype=torch.float32),
        }


n_total    = len(mp_normed)
n_base     = n_total // T_FRAMES if n_total > 32560 else n_total
base_valid = [i for i in range(n_base) if not np.any(np.isnan(mp_normed[i]))]

random.shuffle(base_valid)
split     = int(0.9 * len(base_valid))
train_idx = base_valid[:split]
val_idx   = base_valid[split:]

train_ds = MPLiftDataset(train_idx, mp_normed, xyz_normed, intensity_gt, n_total, augment=True)
val_ds   = MPLiftDataset(val_idx,   mp_normed, xyz_normed, intensity_gt, n_total, augment=False)

BATCH        = 128
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}  Batch: {BATCH}')


## Model — DeformerCARMA
The full three-stage architecture. The Spatial Transformer processes each frame independently over 21 joint tokens. The Temporal Transformer attends across the T=4 frames. The Dynamic Fusion Module predicts forward and backward motion deltas and fuses the frames using learned confidence weights. Two output heads produce the 3D keypoints and the 0–1 openness score.

In [ ]:
class JointEmbedding(nn.Module):
    def __init__(self, d=256):
        super().__init__()
        self.proj     = nn.Linear(3, d)
        self.joint_pe = nn.Embedding(21, d)

    def forward(self, x):
        B   = x.size(0)
        ids = torch.arange(21, device=x.device).unsqueeze(0).expand(B, -1)
        return self.proj(x) + self.joint_pe(ids)


class SpatialTransformer(nn.Module):
    def __init__(self, d=256, nhead=8, n_enc=3, dropout=0.1):
        super().__init__()
        self.embed   = JointEmbedding(d)
        enc          = nn.TransformerEncoderLayer(d_model=d, nhead=nhead, dim_feedforward=d*4,
                                                   dropout=dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers=n_enc, enable_nested_tensor=False)
        self.norm    = nn.LayerNorm(d)
        self.query   = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        dec          = nn.TransformerDecoderLayer(d_model=d, nhead=nhead, dim_feedforward=d*4,
                                                   dropout=dropout, batch_first=True, norm_first=True)
        self.decoder = nn.TransformerDecoder(dec, num_layers=2)

    def forward(self, x):
        tokens  = self.embed(x)
        enc_out = self.encoder(tokens)
        q       = self.query.expand(x.size(0), -1, -1)
        f_plus  = self.decoder(q, enc_out)
        return self.norm(f_plus.squeeze(1))


class TemporalTransformer(nn.Module):
    def __init__(self, d=256, nhead=8, n_enc=3, dropout=0.1):
        super().__init__()
        self.pos_pe  = nn.Embedding(T_FRAMES, d)
        enc          = nn.TransformerEncoderLayer(d_model=d, nhead=nhead, dim_feedforward=d*4,
                                                   dropout=dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers=n_enc, enable_nested_tensor=False)
        self.norm    = nn.LayerNorm(d)

    def forward(self, x):
        B, T, _ = x.shape
        ids  = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        x    = x + self.pos_pe(ids)
        return self.norm(self.encoder(x))


class SimplifiedDFM(nn.Module):
    def __init__(self, d=256, pose_dim=63):
        super().__init__()
        self.pose_head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, pose_dim))
        self.fw_head   = nn.Sequential(nn.Linear(d, d//2), nn.GELU(), nn.Linear(d//2, pose_dim))
        self.bw_head   = nn.Sequential(nn.Linear(d, d//2), nn.GELU(), nn.Linear(d//2, pose_dim))
        self.conf_head = nn.Sequential(nn.Linear(d, 32), nn.GELU(), nn.Linear(32, 1), nn.Sigmoid())

    def forward(self, f_pp):
        B, T, d = f_pp.shape
        theta = self.pose_head(f_pp)
        fw    = self.fw_head(f_pp)
        bw    = self.bw_head(f_pp)
        conf  = self.conf_head(f_pp)

        warped = [theta[:, 0]]
        for t in range(T):
            candidates = [theta[:, t]]
            if t > 0:
                candidates.append(theta[:, t-1] + fw[:, t-1])
            if t < T - 1:
                candidates.append(theta[:, t+1] + bw[:, t+1])
            warped.append(torch.stack(candidates, dim=1).mean(dim=1))
        warped = torch.stack(warped[1:], dim=1)

        fused = (F.softmax(conf, dim=1) * warped).sum(dim=1)
        return fused, theta, fw, bw


class DeformerCARMA(nn.Module):
    def __init__(self, d=256):
        super().__init__()
        self.spatial   = SpatialTransformer(d=d, nhead=8, n_enc=3)
        self.temporal  = TemporalTransformer(d=d, nhead=8, n_enc=3)
        self.dfm       = SimplifiedDFM(d=d, pose_dim=63)
        self.open_head = nn.Sequential(nn.Linear(63, 64), nn.GELU(), nn.Linear(64, 1), nn.Sigmoid())

    def forward(self, x):
        B, T, J, C = x.shape
        f_plus = self.spatial(x.view(B * T, J, C)).view(B, T, -1)
        f_pp   = self.temporal(f_plus)
        fused, theta, fw, bw = self.dfm(f_pp)
        return {
            'kp_pred'  : fused.view(B, 21, 3),
            'open_pred': self.open_head(fused).squeeze(-1),
            'theta'    : theta,
            'fw'       : fw,
            'bw'       : bw,
        }


_m  = DeformerCARMA().to(device)
_x  = torch.randn(2, T_FRAMES, 21, 3, device=device)
_out = _m(_x)
print(f'kp_pred   : {_out["kp_pred"].shape}')
print(f'open_pred : {_out["open_pred"].shape}  range [{_out["open_pred"].min():.3f}, {_out["open_pred"].max():.3f}]')
print(f'Parameters: {sum(p.numel() for p in _m.parameters()):,}')
del _m, _x, _out


## Loss function
Five loss terms: keypoint L2, direct openness MSE, geometric consistency (keypoints and openness head must agree), temporal smoothness, and zero-motion regularization on the DFM. Samples at the open and closed extremes are upweighted to prevent the model from collapsing to the middle of the range.

In [ ]:
def extreme_weight(label):
    dist = torch.min(label, 1.0 - label)
    return 1.0 + 4.0 * (0.5 - dist).clamp(0.0, 0.5) / 0.5


def compute_loss(out, kp_gt, label_gt):
    kp_pred   = out['kp_pred']
    open_pred = out['open_pred']
    theta     = out['theta']
    fw        = out['fw']
    bw        = out['bw']

    w         = extreme_weight(label_gt)
    L_kp      = F.mse_loss(kp_pred, kp_gt)
    L_open    = (w * (open_pred - label_gt).pow(2)).mean()
    L_geo     = (w * (stretch_torch(openness_torch(kp_pred)) - label_gt).pow(2)).mean()
    L_smooth  = (theta[:, 1:] - theta[:, :-1]).pow(2).mean()
    L_motion  = fw.pow(2).mean() + bw.pow(2).mean()

    total = 1.0*L_kp + 2.0*L_open + 0.5*L_geo + 0.1*L_smooth + 0.05*L_motion

    return total, {
        'L_total'  : total.item(),
        'L_kp'     : L_kp.item(),
        'L_open'   : L_open.item(),
        'L_geo'    : L_geo.item(),
        'L_smooth' : L_smooth.item(),
        'L_motion' : L_motion.item(),
        'pred_mean': open_pred.detach().mean().item(),
        'pred_std' : open_pred.detach().std().item(),
    }


## Training
Trains for 20 epochs with AdamW and a cosine schedule with linear warmup. Saves a checkpoint after each epoch and resumes from the last one if the session restarted.

In [ ]:
NUM_EPOCHS   = 20
LR           = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP    = 1.0
WARMUP_STEPS = 200

model     = DeformerCARMA(d=256).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

steps_per_epoch = len(train_loader)
total_steps     = NUM_EPOCHS * steps_per_epoch

def lr_fn(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    prog = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    return max(1e-6/LR, 0.5*(1 + math.cos(math.pi*prog)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_fn)

print(f'Parameters      : {sum(p.numel() for p in model.parameters()):,}')
print(f'Steps per epoch : {steps_per_epoch}   Total: {total_steps}')

start_epoch, global_step = 0, 0
history = {'train_loss':[], 'val_mae':[], 'val_mae_geo':[], 'pred_mean':[], 'pred_std':[]}

ckpts = sorted(glob.glob(str(CKPT_DIR / 'carma_deformer_epoch_*.pth')))
if ckpts:
    ck = torch.load(ckpts[-1], map_location=device)
    model.load_state_dict(ck['model'])
    optimizer.load_state_dict(ck['optimizer'])
    start_epoch = ck['epoch']
    global_step = ck.get('global_step', 0)
    history     = ck.get('history', history)
    print(f'Resumed epoch {start_epoch}  val_MAE={ck["val_mae"]:.4f}')
else:
    print('Starting from scratch.')


@torch.no_grad()
def run_val(model, loader, max_batches=80):
    model.eval()
    maes, maes_geo = [], []
    for i, batch in enumerate(loader):
        if i >= max_batches:
            break
        mp_clip = batch['mp_clip'].to(device)
        label   = batch['label'].to(device)
        out     = model(mp_clip)
        maes.append(F.l1_loss(out['open_pred'], label).item())
        maes_geo.append(F.l1_loss(stretch_torch(openness_torch(out['kp_pred'])), label).item())
    return float(np.mean(maes)), float(np.mean(maes_geo))


best_val_mae = float('inf')

for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    accum = collections.defaultdict(float)
    n, t0 = 0, time.time()

    for batch in train_loader:
        mp_clip = batch['mp_clip'].to(device)
        kp_gt   = batch['kp_gt'].to(device)
        label   = batch['label'].to(device)

        out  = model(mp_clip)
        loss, logs = compute_loss(out, kp_gt, label)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()

        for k, v in logs.items():
            accum[k] += v
        n += 1
        global_step += 1

    val_mae, val_mae_geo = run_val(model, val_loader)
    avg     = {k: v/max(1,n) for k,v in accum.items()}
    is_best = val_mae < best_val_mae
    if is_best:
        best_val_mae = val_mae

    history['train_loss'].append(avg['L_total'])
    history['val_mae'].append(val_mae)
    history['val_mae_geo'].append(val_mae_geo)
    history['pred_mean'].append(avg['pred_mean'])
    history['pred_std'].append(avg['pred_std'])

    print(
        f'Epoch {epoch+1:02d}/{NUM_EPOCHS}  '
        f'loss={avg["L_total"]:.4f}  '
        f'kp={avg["L_kp"]:.4f}  '
        f'open={avg["L_open"]:.4f}  '
        f'smooth={avg["L_smooth"]:.4f}  '
        f'val_MAE={val_mae:.4f}  '
        f'pred={avg["pred_mean"]:.3f}±{avg["pred_std"]:.3f}  '
        f'({time.time()-t0:.0f}s)' + (' <- best' if is_best else '')
    )

    torch.save({
        'epoch': epoch+1, 'global_step': global_step,
        'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
        'val_mae': val_mae, 'history': history,
        'stretch_lo': STRETCH_LO, 'stretch_hi': STRETCH_HI,
    }, str(CKPT_DIR / f'carma_deformer_epoch_{epoch+1:02d}.pth'))

print('Training complete.')


## Training curves
Plots total loss, validation MAE, and prediction distribution over training. The prediction std should grow over time — if it stays near zero the model is collapsing to a single output value.

In [ ]:
epochs_run = list(range(1, len(history['train_loss'])+1))
fig, axes  = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(epochs_run, history['train_loss'], 'b-o', ms=4)
axes[0].set(title='Train Loss', xlabel='Epoch')
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_run, history['val_mae'],     'g-o', ms=4, label='Direct MAE')
axes[1].plot(epochs_run, history['val_mae_geo'], 'm-s', ms=4, label='Geometric MAE')
axes[1].set(title='Validation MAE', xlabel='Epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(epochs_run, history['pred_mean'], 'r-o', ms=4, label='pred mean')
axes[2].plot(epochs_run, history['pred_std'],  'c-s', ms=4, label='pred std')
axes[2].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[2].set(title='Prediction distribution', xlabel='Epoch')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=120)
plt.show()


## Evaluation
Loads the best checkpoint by validation MAE and runs a full evaluation on the held-out set. Reports MAE and correlation for the direct head, geometric head, and ensemble. Also shows a scatter plot and distribution histogram.

In [ ]:
all_ckpts = sorted(glob.glob(str(CKPT_DIR / 'carma_deformer_epoch_*.pth')))
if all_ckpts:
    best = min(all_ckpts, key=lambda p: torch.load(p, map_location='cpu').get('val_mae', 999))
    ck   = torch.load(best, map_location=device)
    model.load_state_dict(ck['model'])
    print(f'Loaded: {best}  val_MAE={ck["val_mae"]:.4f}')


@torch.no_grad()
def full_eval(model, loader, max_batches=300):
    model.eval()
    all_pred, all_geo, all_gt = [], [], []

    for i, batch in enumerate(loader):
        if i >= max_batches:
            break
        mp_clip   = batch['mp_clip'].to(device)
        out       = model(mp_clip)
        kp_pred   = out['kp_pred']
        open_pred = out['open_pred']
        geo_pred  = stretch_torch(openness_torch(kp_pred))

        all_pred.extend(open_pred.cpu().numpy())
        all_geo.extend(geo_pred.cpu().numpy())
        all_gt.extend(batch['label'].numpy())

    pred = np.array(all_pred)
    geo  = np.array(all_geo)
    gt   = np.array(all_gt)
    ens  = (pred + geo) / 2

    for p, name in [(pred,'Direct'), (geo,'Geometric'), (ens,'Ensemble')]:
        mae  = np.mean(np.abs(p - gt))
        corr = np.corrcoef(p, gt)[0,1]
        print(f'{name:12s}  MAE={mae:.4f}  r={corr:.4f}  range=[{p.min():.3f},{p.max():.3f}]')

    print()
    for lo_b, hi_b in [(0,.2),(.2,.4),(.4,.6),(.6,.8),(.8,1.01)]:
        mask = (gt >= lo_b) & (gt < hi_b)
        if mask.sum() == 0:
            continue
        pm, ps = ens[mask].mean(), ens[mask].std()
        print(f'GT [{lo_b:.1f}-{hi_b:.1f}]  n={mask.sum():5d}  pred={pm:.3f}±{ps:.3f}')

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].scatter(gt, ens, s=3, alpha=0.2, color='steelblue')
    axes[0].plot([0,1],[0,1],'r--', lw=1.5, label='perfect')
    axes[0].set(xlabel='GT intensity', ylabel='Ensemble intensity', title='Ensemble predicted vs GT')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].hist(ens, bins=50, color='darkorange', alpha=0.7, label='predictions', density=True)
    axes[1].hist(gt,  bins=50, color='steelblue',  alpha=0.5, label='GT labels',   density=True)
    axes[1].set(xlabel='Intensity', title='Prediction vs GT distribution')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/evaluation.png', dpi=120)
    plt.show()


full_eval(model, val_loader)


## Inference
Runs the full pipeline on a single image — MediaPipe detection, landmark normalization, Deformer lifting, and intensity estimation. Shows the 2D skeleton, lifted 3D pose, and the three intensity bars side by side.

In [ ]:
def infer_image(img_path_or_array, model, stretch_lo=STRETCH_LO, stretch_hi=STRETCH_HI):
    model.eval()

    if isinstance(img_path_or_array, str):
        img_np = np.array(Image.open(img_path_or_array).convert('RGB'))
    else:
        img_np = img_path_or_array

    base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
    hand_options = mp_vision.HandLandmarkerOptions(
        base_options=base_options, num_hands=1, min_hand_detection_confidence=0.3)
    detector = mp_vision.HandLandmarker.create_from_options(hand_options)
    mp_img   = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_np)
    result   = detector.detect(mp_img)
    detector.close()

    if not result.hand_landmarks:
        print('No hand detected.')
        return None

    lm_raw  = np.array([[l.x, l.y, l.z] for l in result.hand_landmarks[0]], dtype=np.float32)
    lm_norm = normalize_mp_landmarks(lm_raw)
    lm_clip = np.stack([lm_norm] * T_FRAMES, axis=0)
    inp     = torch.tensor(lm_clip, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        out = model(inp)

    direct   = float(out['open_pred'][0].cpu())
    raw_geo  = float(openness_torch(out['kp_pred'])[0].cpu())
    geo      = float(np.clip((raw_geo - stretch_lo) / max(stretch_hi - stretch_lo, 1e-6), 0.0, 1.0))
    ensemble = (direct + geo) / 2.0
    state    = 'CLOSED' if ensemble > 0.5 else 'OPEN'

    sk = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),(0,9),(9,10),(10,11),(11,12),
          (0,13),(13,14),(14,15),(15,16),(0,17),(17,18),(18,19),(19,20),(5,9),(9,13),(13,17)]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_np)
    axes[0].axis('off')
    H, W = img_np.shape[:2]
    for a, b in sk:
        axes[0].plot([lm_raw[a,0]*W, lm_raw[b,0]*W], [lm_raw[a,1]*H, lm_raw[b,1]*H], 'lime', lw=1.5)
    axes[0].scatter(lm_raw[:,0]*W, lm_raw[:,1]*H, s=20, c='red', zorder=5)
    axes[0].set_title('MediaPipe Detection')

    kp_np = out['kp_pred'][0].cpu().numpy()
    ax3   = fig.add_subplot(1, 3, 2, projection='3d')
    ax3.scatter(*kp_np.T, c='dodgerblue', s=30)
    for a, b in sk:
        ax3.plot(*zip(kp_np[a], kp_np[b]), color='gray', lw=1.2)
    ax3.set_title('Lifted 3D (Deformer)')

    ax = axes[2]
    ax.axis('off')
    ax.set_xlim(0,1)
    ax.set_ylim(0,1)
    for (name, val, clr), y0 in zip([('Direct',direct,'steelblue'),('Geo',geo,'darkorange'),('Ensemble',ensemble,'darkgreen')],[0.62,0.38,0.12]):
        ax.add_patch(plt.Rectangle((0.05,y0), 0.90, 0.18, color='#e8e8e8'))
        ax.add_patch(plt.Rectangle((0.05,y0), 0.90*val, 0.18, color=clr, alpha=0.8))
        ax.text(0.50, y0+0.22, name, ha='center', fontsize=9, color='gray')
        ax.text(0.50, y0+0.08, f'{val:.3f}', ha='center', fontsize=13, fontweight='bold', color=clr)
    ax.text(0.05, 0.02, '0=Open', ha='left', fontsize=8, color='gray')
    ax.text(0.95, 0.02, '1=Fist', ha='right', fontsize=8, color='gray')
    ax.text(0.50, 0.93, state, ha='center', fontsize=13, fontweight='bold',
            color='firebrick' if ensemble > 0.5 else 'forestgreen')
    ax.set_title('CARMA Hand Intensity')

    plt.tight_layout()
    plt.savefig('/content/inference_result.png', dpi=120)
    plt.show()

    print(f'Direct  : {direct:.4f}')
    print(f'Geo     : {geo:.4f}')
    print(f'Ensemble: {ensemble:.4f}  -> {state}')
    return {'direct': direct, 'geo': geo, 'ensemble': ensemble}


sample_path = RGB_DIR / f'{val_idx[0]:08d}.jpg'
result = infer_image(str(sample_path), model)
if result:
    print(f'GT label: {intensity_gt[val_idx[0]]:.4f}')
